**PART 4.  EMBEDDINGS**

One-hot / Bag-of-Words columns say nothing about meaning.  An EMBEDDING gives each
token a short list of numbers (a DENSE vector, e.g. 2 numbers here, 300 or 4,096 in
real models) that the model LEARNS, so that similar words get similar vectors.

  - An embedding layer is just a TABLE with one row per vocabulary id:
        nn.Embedding(vocab_size, dim)   -> a weight matrix of shape (vocab_size, dim)
    Looking up id 4 means "take row 4".  It is learned by gradient descent like any
    other weight (Day 14 and 15).
  - Similarity of meaning = cosine similarity of vectors.
  - A SENTENCE vector (simple version): the AVERAGE of its word vectors.

  Pretend these 2-number vectors were learned:

        the       [ 0.1,  0.0]        movie     [ 0.5,  0.5]
        was       [ 0.0,  0.1]        acting    [ 0.4,  0.2]
        great     [ 0.0,  1.0]        terrible  [ 0.0, -1.0]

  A tiny classifier on top:  z = w . x + b  with  w = [0.0, 2.0],  b = -0.5,
  then  p(positive) = 1 / (1 + e^(-z))   (the sigmoid of Day 14).

  EX 5 (paper, 15 min).

     a) Sentence vector of d1 (the average of the 4 word vectors), then z and p.
       * Sentence vector: (0.1+0.5+0.0+0.0/4 + 0.0+0.5+0.1+0.1) = [0.15, 0.40]
       * z = (0.0 x 0.15) + (2.0 x 0.40) - 0.5 = 0.3
       * p = 1/1+e^0.3 = 0.5744

     b) The same for d2 ("terrible" instead of "great").
       * Sentence vector: (0.1+0.5+0.0+0.0/4 + 0.0+0.5+0.1-0.1) = [0.15, -0.10]
       * z = (0.0 x 0.15) + (2.0 x -0.10) - 0.5 = -0.7
       * p =  1/1+e^-0.7 = 0.3318

     c) In one sentence: which number made the difference between d1 and d2, and
        why is that the "learning" of an embedding?
       * The second coordinate changed from 1.0 (for great) to -1.0 (for terrible), which represents the semantic "sentiment polarity" dimension automatically discovered and learned by the embedding space to separate positive words from negative ones. 

     d) cos(movie, acting)  with the vectors above.  (Round to 2 decimals.)
       * Vectors: movie = [0.5, 0.5], acting = [0.4, 0.2]
       * Dot Product: (0.5 x 0.4) + (0.5 x 0.2) = 0.2 + 0.1 = 0.3
       * movie = sqrt(0.5^2 + 0.5^2) = sqrt(0.5) = 0.7071
         acting = sqrt(0.4^2 + 0.2^2) = sqrt(0.2) = 0.4472
       * cos = 0.3/0.7041*0.4472 = 0.95

  Why embeddings beat Bag of Words: "great" and "excellent" can end up close
  together, so a model that saw only "great" in training can still handle
  "excellent".  Pretrained embeddings (Word2Vec, GloVe) and the embeddings inside LLMs
  are learned from billions of words.  Day 22 uses them for semantic search.

**PART 5.  SEQUENCE LENGTH AND PADDING**

A batch is a rectangle, but sentences have different lengths.  Fix: choose a
SEQUENCE LENGTH L (the longest sentence in the batch, or a cap).  Shorter sentences
are PADDED at the end with <PAD> (id 0); longer ones are TRUNCATED.  The model must
be told which positions are padding (an ATTENTION MASK: 1 = real token, 0 = pad),
otherwise it would learn from the filler.

  EX 6 (paper, 5 min).  Write d1 to d4 as id lists padded to length 4, and write the
  mask for d3.   (Use the ids from EX 1.)

   * "the movie was great": [6, 4, 7, 3] (Already length 4, no padding needed) 
   * "the movie was terrible": [6, 4, 7, 5] (Already length 4, no padding needed)
   * "great great acting": [3, 3, 2, 0] (Length 3, padded with 0 at the end)
   * "the acting was terrible": [6, 2, 7, 5] (Already length 4, no padding needed)

   * mask for d3: 1 = real token, 0 = padding -> [1, 1, 1, 0]

  Cost: attention (Part 6) compares every token with every other, so the work grows
  with L squared.  This is why LLMs have a limited CONTEXT WINDOW (Day 21).


**PART 6.  THE IDEA OF ATTENTION**

Problem: in "the bank of the river", the meaning of "bank" depends on "river".  A
Bag of Words cannot see that.  ATTENTION lets each token LOOK at all the other tokens
and decide how much to use each one.

  1. For one token, compute a SCORE against every token (how relevant is it?).
  2. Turn the scores into WEIGHTS that are positive and add up to 1 with SOFTMAX:
            weight_i = e^(score_i) / (e^(score_1) + e^(score_2) + ...)
  3. The new vector of the token = the weighted sum of the vectors of all tokens.

  Worked: scores [2, 1, 0]
        e^2 = 7.389 ; e^1 = 2.718 ; e^0 = 1 ;  sum = 11.107
        weights = [0.665, 0.245, 0.090]            (they add up to 1)

  EX 7 (paper, 10 min).

     a) softmax of [0, 0, 0]
       * e^0 = 1, [1/3, 1/3, 1/3] = [0.3333, 0.3333, 0.3333, 0.3333]

     b) softmax of [3, 0, 0]
       * e^3 = 20.0855, e^0 = 1, [20.0855/22.0855, 1/22.0855, 1/22.0855] = [0.909, 0.045, 0.045]

     c) The three token vectors are  [1,0], [0,1], [1,1].  With the weights of the
        worked example, compute the weighted sum  0.665x[1,0] + 0.245x[0,1] + 0.090x[1,1].
       * First coordinate: (0.665 x 1) + (0.245 x 0) + (0.090 x 1) = 0.665 + 0.0 + 0.090 = 0.755
       * Second coordinate: (0.665 x 0) + (0.245 x 1) + (0.090 x 1) = 0.0 + 0.245 + 0.090 = 0.335
       * = [0.755, 0.335]

     d) What does a very large score do to the other weights?
       * A very large score completely dominates the exponential function (e^score), driving its resulting weight close to 1.0. Because all weights must sum to 1, this forces all the other weights to shrink drastically and approach 0.0 (as seen in part (b) where a score of 3 pushed the other weights down to 0.045). In an attention mechanism, this allows a token to focus almost exclusively on one single relevant word while ignoring everything else.

  Day 20 gives these scores a name (query, key, value).  Today you only need the idea:
  scores -> softmax -> weighted sum.


**CODE EXERCISE (EX 8, 20 minutes)  - plain Python, loops and dictionaries**

In [3]:
docs = ["the movie was great", "the movie was terrible",
          "great great acting", "the acting was terrible"]

def build_vocab(docs):
    words = set()
    for d in docs:
        for w in d.split():
            words.add(w)
    vocab = {}
    for i, w in enumerate(sorted(words)):
        vocab[w] = i + 2
    return vocab

def to_ids(doc, vocab, length):
    ids = [vocab.get(w, 1) for w in doc.split()]
    return ids + [0] * (length - len(ids))

def bow(doc, vocab):
    vec = [0] * len(vocab)
    for w in doc.split():
        vec[vocab[w] - 2] += 1
    return vec

In [4]:
vocab = build_vocab(docs)
print(vocab)
print([to_ids(d, vocab, 4) for d in docs])
print([bow(d, vocab) for d in docs])
print(to_ids("the film was great", vocab, 4))

{'acting': 2, 'great': 3, 'movie': 4, 'terrible': 5, 'the': 6, 'was': 7}
[[6, 4, 7, 3], [6, 4, 7, 5], [3, 3, 2, 0], [6, 2, 7, 5]]
[[0, 1, 1, 0, 1, 1], [0, 0, 1, 1, 1, 1], [1, 2, 0, 0, 0, 0], [1, 0, 0, 1, 1, 1]]
[6, 1, 7, 3]


**QUESTIONS FOR YOUR NOTES (Q1 to Q8): Hours 1 and 2**

- **Q1  What is a token?  Is a token always a word?**
    * A token is a piece of text that the model reads as one unit. It is not always a word: with subword tokenizers (GPT-style models) it can be a whole word, part of a word or a single character.
- **Q2  What are `<PAD>` and `<UNK>` for?**
    * `<PAD>` (id 0) fills short sequences up to equal length so a batch is a rectangle. `<UNK>` (id 1) stands for any word that is not in the vocabulary, so the model still gets an input.
- **Q3  Give two weaknesses of Bag of Words.**
    * Word order and negation are lost ("dog bites man" = "man bites dog").
    * The vectors are as long as the vocabulary and mostly zeros, and synonyms ("great", "excellent") are unrelated columns. Filler words count as much as informative ones.
- **Q4  In TF-IDF, why do words like "the" get a small weight?**
    * "the" appears in almost every document, so df is large and IDF = ln(N / df) is close to 0 (ln(4/3) = 0.2877, against ln(4/2) = 0.6931 for a word in only 2 documents). A word is important only if it is frequent in this document but rare in the others.
- **Q5  What is an embedding, and what is the shape of an embedding layer?**
    * An embedding is a short dense vector of learned numbers for each token, so that similar words get similar vectors. The embedding layer is a table with one row per vocabulary id: shape (vocab_size, dim), for example `nn.Embedding(3332, 32)` has 106,624 weights.
- **Q6  Why do sentences need padding, and what is the attention mask for?**
    * A batch must be a rectangle, but sentences have different lengths, so short ones are padded with `<PAD>` (and long ones truncated). The mask (1 = real token, 0 = pad) tells the model which positions are real, so the filler is ignored (in AvgEmb it zeros the pad vectors and divides by the number of real tokens).
- **Q7  In one sentence: what does attention do for a token?**
    * It lets the token look at all the other tokens, give each one a weight by relevance (softmax of the scores), and build its new vector as the weighted sum of them.
- **Q8  Why does a longer context window cost much more?**
    * Attention compares every token with every other token, so the score table is n x n and the cost grows with the square of the length: twice as many tokens means four times the work.